### Step 1: Import Libraries and load the environment variables

In [22]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pprint import pprint
import gradio as gr
import json


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

client = OpenAI()

### Step 2: Simple RAG w/ Guardrails & Dynamic Context Injection

In [23]:
#These guardrails appear to work better

system_message = """You are a digital twin of Andrea Gosset Soto. When people talk to you, 
you respond AS Andrea Gosset Soto - in first person, using her voice, personality, and knowledge. 
Start your very first message as: "Hi there! I'm Andrea" and go on with your regular message.

Important: d not make things up. If you don't know an answer, say you don't know.
The only factual information available to you is what's in this system message.
You cannot get any more factos about Andrea from the interenet or make them up.

The ONLY factual information about Andrea you can use is between the *** markers.
If you don't know the answer to a question based on that info, say you don't know.
If a question is aked that is not answerable based on that info, say you don't know.

***

Andrea is an experienced Front End Engineer. She's based in Atlanta, Georgia. She has a BS in 
Animation and Visual Effects from the Mexican university Tecnologico de Monterrey which she started in
2011 and finished in 2015.

She's been a Front End Engineer since 2016 creating web applications and managing 
CMS across various industries. Proven team leader with a strong commitment to understanding business 
aspects for innovative solutions, showcasing skills in mentorship, continuous learning, and optimizing 
user experience through modern technologies.

Other career history:
2015: Intern at MetaCube as a 3D Digital Modeller for the Dia de Muertos movie
2016-2018: Front End Engineer at Base22
2018-2022: Senior Front End Engineer at Globant
    - Led a consulting team to develop two intranets alongside the client Realogy using Angular 7, NgRx, 
    Jasmine, and Apollo GraphQL, resulting in realtors benefiting from a centralized platform for 
    accessing their essential day-to-day tools.
    - Engineered an npm library with Node to seamlessly integrate a standardized theme across various 
    intranets and applications, improving consistency, efficiency, and fostering a more unified and 
    cohesive visual identity.
    - Collaborated with Stanley Black and Decker's team to enhance their Site Manager and Public API 
    intranets. Developed a comprehensive week-to-week action plan streamlining their development and 
    Scrum processes.
    - Mentored 3 peers at a time to facilitate their career advancement within their teams, providing 
    guidance and support.
    - Conducted thorough interviews with prospective candidates for seamless integration into the company.

2022-2023: Software Engineer at Twitter
    - Directed the front-end effort for the Semantic Core UI migration leveraging React.js, Typescript, 
    Javascript, and CSS enabling clients to efficiently gather insights for optimal ad promotion within 
    Twitter's internal platform.
    - Mentored and spearheaded a 5-person division within the TwST (Twitter Security Team) department, 
    fostering professional growth, team cohesion, and cross-functional collaboration to achieve key 
    departmental objectives.
    - Authored the Technical Design Document for the Semantic Core UI migration to accommodate new 
    implementations, enhance code quality, and improve maintainability to ensure an organized and 
    well-architectured development.

2023-Present: Senior Front End Engineer at Rooms To Go
    Improved website performance, streamlined content updates, and collaborated with cross-functional 
    teams to implement key features. Additionally led the front-end team, optimizing Agile processes 
    to enhance efficiency and delivery.
    - Spearheaded front-end artificial intelligence (AI) projects, enhancing customer engagement and 
    contributing to a 30 percent boost in sales revenue.
    - Migrated the main E-commerce webpage from React.js and MUI to Next.js, TypeScript, and Tailwind, 
    implementing a more efficient server-side solution that resulted in a 45% performance improvement, 
    enhancing the user experience.
    - Developed, updated, and revamped React.js components, Strapi schemas, and the organization's npm 
    library to streamline E-Commerce content updates, ensuring timely refreshes that featured weekly 
    sales and drove increased revenue.
    - Collaborated with the Content, UX and Marketing teams to gather and comprehend new requirements, 
    translating them into actionable tickets while assuming leadership responsibilities for their 
    implementation within the front-end team.


Technical Skills:
Javascript (ES6+), React.js, Typescript, Next.js, Angular 7, Vue, jQuery, Jest, Jasmine, Cypress, 
Redux, NgRx, HTML/CSS, Mustache, Handlebars, SASS, MUI, Tailwind, Apollo GraphQL, REST APIs, n8n, 
Claude, Cursor, Gemini, Webpack, Gulp, Grunt, Node, Scala, Strapi, IBM WCM, Liferay DXP, Git, Agile, 
Scrum, Kanban.

Languages:
English (fluent), Spanish (native), French (proficient).

What drives her: She loves to dive into the business behind the ask as well as finding areas that could
use some improvement in our code as well as in our Sprint processes and create strategies to address 
them in a way that is subtle an easy to manage for the whole team. She also loves helping her coworkers by
being a mentor to them and help them grow.

Her approach: Always thinking two steps ahead and thinking as a mentor.

Communication style:Friendly and accessible, thinking as a mentor.


***
"""

In [24]:
Topic_Context = {
    "2008": "***In 2008 Andrea was studying highshool abroad***",
    "cooking": "***Andrea doesn't like cooking***",
    "peaches": "***Andrea loves peaches, she likes them by themselves, on cakes, on jello, \
        with cinamon, etc***",
    "singing": "***Andrea has loved singing since she was a kid, previously she only did it \
        in the shower but as she grew older she started doing it while driving and almost everywhere she \
        could.. Now she's been taking singing lessons online with a great and kind teacher that has \
        helped her improve her technique and avoid hurting her vocal chords. She's preparing 10 songs (5 \
        in english and 5 in spanish) all different genres to have on her repertoire.***"
}

### Step 3: Add tool-calling functionality (Pushover)

In [25]:
PUSHOVER_USER = os.getenv("PUSHOVER_USER")
PUSHOVER_TOKEN = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [26]:
#Create send_notification function
import requests

def send_notification(message: str):
    payload = {"user": PUSHOVER_USER, "token": PUSHOVER_TOKEN, "message": message}
    requests.post(pushover_url, data=payload)

In [27]:
#Describe Pushover as an LLM tool
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the real-world of you via Pushover on mobile. Use this if the user needs to alert the real-world version of you.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device."
            }
        },
        "required": ["message"]
    }
}

In [28]:
#Add Pushover to the list of tools for the LLM
tools = [{"type": "function", "function": send_notification_function}]

### Step 2b & 3b & 4b: Create new function, describe it, add it to the list of tools

In [29]:
import random

def dice_roll():
    result = random.randint(1,6)
    return result

roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die for games, decisions, or random number generation.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

tools.append({"type": "function", "function": roll_dice_function})


In [30]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        if function_name == "send_notification":
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
        #elif function_name == "insert_function_name_3":
            # content = insert_function_name3(args["message"])
        #....
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id,
        }

        tool_results.append(tool_call_result)
        
    return tool_results

In [33]:

def response_ai(message, history):
    #Inject dynamic context based on keyworkds in the message
    system_message_enhanced = system_message
    messages = [
        {"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}
    ]
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + context

    #As usual but we substitute system_message for system_message_enhanced
    #messages = messages
    #print(system_message_enhanced) #Debugging line to see the final system message
    #print("System message used for this response: \n", system_message_enhanced) #Debugging line to see the final system message
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )

    #Check if model wants to call a tool
    message = response.choices[0].message

    while message.tool_calls:
        from pprint import pprint
        pprint(f"print from message tool calls: {message.tool_calls}")
        tools_result = handle_tool_call(message.tool_calls)
        messages.append(message)
        messages.extend(tools_result)

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools
        )
        message = response.choices[0].message
        reply = response.choices[0].message.content

        print(f"print from inside the while: {reply}")
    return message.content

In [ ]:

def response_ai(message, history):
    #Inject dynamic context based on keyworkds in the message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + context

    #As usual but we substitute system_message for system_message_enhanced
    messages = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]
    #print(system_message_enhanced) #Debugging line to see the final system message
    #print("System message used for this response: \n", system_message_enhanced) #Debugging line to see the final system message
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message
    reply = response.choices[0].message.content

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)

        #Actually send the notification
        send_notification(args["message"])
        print(f"Sent notification: {args['message']}")
    else:
        print(message.content)

    return reply

In [34]:
gr.ChatInterface(fn=response_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7929
* To create a public link, set `share=True` in `launch()`.


('print from message tool calls: '
 "[ChatCompletionMessageFunctionToolCall(id='call_BpwAdkZcENECf8BVdvmEgfJk', "
 "function=Function(arguments='{}', name='dice_roll'), type='function'), "
 "ChatCompletionMessageFunctionToolCall(id='call_SPXoMtxqpXmhOBjBAltouiNk', "
 "function=Function(arguments='{}', name='dice_roll'), type='function')]")
print from inside the while: None
('print from message tool calls: '
 "[ChatCompletionMessageFunctionToolCall(id='call_JkBSDCfezInqffSvAjtRpEhr', "
 'function=Function(arguments=\'{"message":"The highest rolled dice value is '
 '3."}\', name=\'send_notification\'), type=\'function\')]')
print from inside the while: I rolled two dice, and the results were 3 and 2. I have sent a notification to the real Andrea with the highest rolled value, which is 3. Is there anything else you'd like to do?
